# Proyecto medio: clasificación de enfermedades en hojas de café (RoCoLe)
Dataset: [RoCoLe](https://www.kaggle.com/datasets/nirmalsankalana/rocole-a-robusta-coffee-leaf-images-dataset) (1560 fotos, 3 clases: `healthy`, `rust`, `red_spider_mite`).

1. **Preparación de datos**: partición 70/15/15 sin leakage, data generator, GAN.
2. **Hiperparámetros**: Hyperband.
3. **Validación cruzada**: K-fold y variantes.
4. **Mejoras y evaluación final**: pesos de clase, fine-tuning y GAN, comparadas en validación; test se usa una sola vez.

In [ ]:
import os, sys
REPO = "/content/ia-Kaggle-Coffee"   # Colab: el notebook se clona solo si falta
if os.path.exists("/content") and not os.path.isdir(REPO):
    !git clone -q -b claude/brave-archimedes-jnuxk1 https://github.com/JavierAnd0/ia-Kaggle-Coffee {REPO}
if os.path.isdir(REPO):
    %cd {REPO}
sys.path.insert(0, os.getcwd())
!pip -q install keras-tuner   # Colab ya trae TensorFlow, sklearn y pandas
import pandas as pd, numpy as np, tensorflow as tf
from src import config, data_prep, generators, tuning, cross_val, gan, train as trainer
tf.keras.utils.set_random_seed(config.SEED)

FAST_MODE = False        # True = prueba rápida (pocas épocas); False = entrenamiento completo
RUN_HYPERBAND = False    # False = reutiliza results/best_hp.json (ya calculado, ~1h15m)
RUN_CV = False           # False = reutiliza results/cv_results.csv (ya calculado, ~1h)
config.FAST_MODE = FAST_MODE
EPOCHS, FT_EPOCHS = (2, 1) if FAST_MODE else (30, 10)
GAN_EPOCHS, N_SYNTH = (5, 20) if FAST_MODE else (200, 150)

## 0. Datos
Etiqueta = carpeta; planta = prefijo del archivo (`C10P10E1.jpg` → `C10P10`, 390 plantas).

In [ ]:
root = config.ROOT / "data"
df = data_prep.build_dataframe(
    root,
    label_map={"coffee___healthy": "healthy", "coffee___rust": "rust",
               "coffee___red_spider_mite": "red_spider_mite"},
    group_regex=r"^(C\d+P\d+)")
df = data_prep.add_file_hash(df)
data_prep.eda(df)

## Topic 1 – Preparación
### Partición 70/15/15 sin data leakage
Se agrupa por planta y por duplicados exactos (hash) **antes** de cualquier augmentation o GAN; los splits se guardan en `data_splits/`.

In [ ]:
train, val, test = data_prep.split_dataset(df)
data_prep.assert_no_leakage(train, val, test)
pd.concat({k: v['label'].value_counts() for k, v in dict(train=train, val=val, test=test).items()}, axis=1)

### Data generator (carga por lotes con `tf.data`)

In [ ]:
li = generators.make_label_index(train['label'])
ds = generators.make_dataset(train, li, training=True, augment=True)
x, y = next(iter(ds)); print(x.shape, y.shape, li)

### GAN condicional
Se entrena **solo con imágenes de train** (64×64). Luego se generan hojas sintéticas de la clase minoritaria, que solo se añaden al entrenamiento; val y test son siempre reales.

In [ ]:
minority = train['label'].value_counts().idxmin()
imgs = gan.load_images(train)
cgan = gan.CGAN(len(li))
cgan.fit(imgs, train['label'].map(li).values, epochs=GAN_EPOCHS)
synth = gan.make_synthetic_df(cgan, minority, li[minority], N_SYNTH, config.SYNTH_DIR)
print(minority, '-> sintéticas:', len(synth))

In [ ]:
import matplotlib.pyplot as plt
real = train[train.label == minority].sample(8, random_state=0)['filepath'].tolist()
fig, ax = plt.subplots(2, 8, figsize=(16, 4))
for a, p in zip(ax[0], real): a.imshow(tf.keras.utils.load_img(p, target_size=(64, 64))); a.axis('off')
for a, p in zip(ax[1], synth['filepath'][:8]): a.imshow(tf.keras.utils.load_img(p, target_size=(64, 64))); a.axis('off')
ax[0][0].set_title('reales'); ax[1][0].set_title('sintéticas'); plt.show()

## Topic 2 – Hyperband
La búsqueda ya se ejecutó (90 trials) y sus resultados están en `results/best_hp.json`.

In [ ]:
if RUN_HYPERBAND or not (config.RESULTS_DIR / "best_hp.json").exists():
    best = tuning.run_hyperband(train, val)
else:
    best = tuning.load_best_hp()
best.values

## Topic 3 – Validación cruzada (train+val; test intacto)
- **K-fold / Stratified K-fold / Group Stratified K-fold / Hold-out / Monte Carlo**: ejecutados.
- **Leave-p-out / Leave-one-out**: requieren C(n,p) / n entrenamientos; con n=1336 y una CNN es inviable.
- **Time-series (rolling)**: no aplica, las imágenes no tienen orden temporal.

In [ ]:
from math import comb
trval = pd.concat([train, val]).reset_index(drop=True)
n = len(trval); print('LOO:', n, 'entrenamientos | Leave-2-out:', comb(n, 2))
data_prep.make_folds(trval)
if RUN_CV or not (config.RESULTS_DIR / "cv_results.csv").exists():
    cv = cross_val.run_cv(best, trval)
else:
    cv = pd.read_csv(config.RESULTS_DIR / "cv_results.csv")
cv

## Mejoras: pesos de clase, fine-tuning y GAN
La CV mostró accuracy ≈ 0.72–0.75 pero F1 macro ≈ 0.59: la clase `red_spider_mite` (la menos frecuente) se clasifica mal. Se comparan, **sobre validación** y con la misma semilla, cuatro variantes acumulativas:
1. modelo base, 2. + pesos de clase, 3. + fine-tuning de las últimas capas, 4. + imágenes sintéticas de la GAN.
   - **4b (control):** en vez de la GAN, repite imágenes *reales* de la clase minoritaria (sobremuestreo). Sirve para saber si la GAN aporta algo más que balancear.

Los pesos de clase se calculan siempre con los datos reales de train, para que las sintéticas no alteren ese efecto. Es una sola corrida por variante, así que diferencias pequeñas pueden ser ruido. Las sintéticas son de baja resolución (64×64 ampliadas): el clasificador podría aprender «borroso = clase minoritaria»; el control 4b ayuda a detectarlo.

In [ ]:
abl, models = trainer.run_experiments(best, train, val, li, synth_df=synth,
                                    epochs=EPOCHS, finetune_epochs=FT_EPOCHS)
abl.to_csv(config.RESULTS_DIR / "ablation_val.csv")
abl.round(3)

## Evaluación final en test (una sola vez)
Se elige la variante con mejor F1 macro en **validación**.

In [ ]:
from sklearn.metrics import classification_report
best_name = abl['f1_macro'].idxmax(); print('Variante elegida:', best_name)
ev = trainer.evaluate(models[best_name], best, test, li)
rep = classification_report(ev['y_true'], ev['y_pred'], target_names=sorted(li, key=li.get), zero_division=0)
print(rep)
cm = trainer.confusion_df(ev, li); print(cm)
(config.RESULTS_DIR / "test_report.txt").write_text(f"Variante: {best_name}\n\n{rep}\n{cm}\n")
cm

## Conclusiones
_(completar con los resultados: tabla de CV, comparación de variantes en validación y reporte final en test; discutir el efecto de la GAN, la clase minoritaria y las limitaciones: test con solo 21 imágenes de `red_spider_mite`, una corrida por variante.)_